# Part B — Predictable Password Reset Token Attack

This demonstration simulates an application that seeds Python's `random.Random` (MT19937) with the current Unix timestamp in integer seconds and generates 16 consecutive 8-bit outputs. The resulting 16-byte reset token is displayed and recorded in hexadecimal. The attacker searches all timestamps from 300 seconds before through 300 seconds after the approximate generation time.

The implementation intentionally uses the same standard-library MT19937 and byte extraction as Part A: `bytes(generator.getrandbits(8) for _ in range(16))`. This is an insecure educational example, not a way to generate real reset tokens.

## Why it is vulnerable

A timestamp is not a secret. An attacker who knows roughly when a reset was requested can enumerate the small set of possible second-based seeds, regenerate each candidate token, and compare it with the observed value. Seeding a generator with a timestamp makes its initial state guessable.

MT19937 is designed for simulation, not cryptographic security. Its output is deterministic and is not designed to resist an adversary who can observe or predict its output. Use a cryptographically secure source such as Python's `secrets.token_bytes(16)` for security tokens.

## Prediction pseudocode

```text
generation_time = current Unix time in integer seconds
token = concatenate(MT19937(generation_time).getrandbits(8) for 16 bytes)
save generation_time and token as hexadecimal values

for candidate_time from max(0, approximate_time - 300)
                      through approximate_time + 300, inclusive:
    candidate_token = concatenate(MT19937(candidate_time).getrandbits(8)
                                  for 16 bytes)
    compare candidate_token with observed_token; remember a match
report candidates tested and recovered timestamp
regenerate token from recovered timestamp and verify equality
```

In [1]:
from pathlib import Path
import sys

PART_B_DIR = Path.cwd()
if not (PART_B_DIR / "password_reset_token_attack.py").is_file():
    PART_B_DIR = PART_B_DIR / "part_b"
sys.path.insert(0, str(PART_B_DIR))

from password_reset_token_attack import run_attack

## Generate the token and recover its timestamp

Run the next cell to generate a fresh current-time token, save its generation timestamp and hexadecimal token to `password_reset_token_record.json`, exhaustively test the ±300-second interval, then regenerate and verify the recovered token. The code counts each tested second, including candidates after a match.

In [2]:
result = run_attack()
start, end = result["search_interval"]
print(f"Original timestamp: {result['generation_timestamp']} (0x{result['generation_timestamp']:x})")
print(f"Original token (hex): {result['token_hex']}")
print(f"Saved record: {Path(result['record_path']).name}")
print(f"Search interval: [{start}, {end}]")
print(f"Candidates tested: {result['candidates_tested']}")
print(f"Recovered timestamp: {result['recovered_timestamp']} ({result['recovered_timestamp_hex']})")
print(f"Regenerated token (hex): {result['regenerated_token_hex']}")
print(f"Verification: {result['verified']}")

Original timestamp: 1793960107 (0x6aeda8ab)
Original token (hex): a4234efca6d57fd7b124152356afbe52
Saved record: password_reset_token_record.json
Search interval: [1793959807, 1793960407]
Candidates tested: 601
Recovered timestamp: 1793960107 (0x6aeda8ab)
Regenerated token (hex): a4234efca6d57fd7b124152356afbe52
Verification: True


The captured output above is an example run; rerunning the attack cell uses the current Unix time and updates the saved record. The search interval is inclusive and the implementation tests the full interval even after finding the token.